# Choosing the severity grid (Section 6.3)

Shows what each corruption does at each severity level, so the values in `configs/severity_grid.yaml` can be chosen with open eyes.

**Rule:** this notebook only ever loads **training** patients. The held-out vendor B patients and vendors A, C and D are not looked at, so the grid cannot be tuned towards the result.

Needs `configs/split.json` (run `python -m ssp.split` first) and the `MNMS_ROOT` environment variable, or set `ROOT` below.
On Colab, run cells 2 and 3 of `00_colab_runner.ipynb` first, in the same session.

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ssp import config, corruptions, data, paths

ROOT = None                      # or a path, e.g. "data/mock"
root = paths.data_root(ROOT)
cases = data.discover_cases(root).set_index("patient", drop=False)
split = config.split()
grid = config.severity_grid()
train_patients = split["train"][:5]
print("Grid status:", grid["status"], "| corruptions:", list(grid["corruptions"]), "| levels:", grid["levels"])
print("Looking at training patients:", train_patients)

In [ ]:
# One mid-ventricular slice, every corruption at every level
case = cases.loc[train_patients[0]]
image, label, spacing = data.load_volume(case, "ED")
z = int(np.argmax(label.sum(axis=(0, 1))))          # the slice with the most labelled heart
lo, hi = np.percentile(image, [0.5, 99.5])

names, S = list(grid["corruptions"]), grid["levels"]
fig, axes = plt.subplots(len(names), S + 1, figsize=(2.3 * (S + 1), 2.4 * len(names)))
for r, name in enumerate(names):
    axes[r, 0].imshow(image[:, :, z].T, cmap="gray", vmin=lo, vmax=hi)
    axes[r, 0].set_title("original", fontsize=9)
    axes[r, 0].set_ylabel(name, fontsize=10)
    for s in range(1, S + 1):
        bad = corruptions.corrupt(image, spacing, name, s, grid, key=f"{case['patient']}|ED")
        value = grid["corruptions"][name]["values"][s - 1]
        axes[r, s].imshow(bad[:, :, z].T, cmap="gray", vmin=lo, vmax=hi)
        axes[r, s].set_title(f"level {s}  ({value})", fontsize=9)
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
# How much each level changes the image, averaged over the training patients above.
# Unit: mean absolute change as a percentage of the image's intensity range.
rows = []
for p in train_patients:
    case = cases.loc[p]
    for phase in data.PHASES:
        image, _, spacing = data.load_volume(case, phase)
        rng_ = float(np.percentile(image, 99.5) - np.percentile(image, 0.5)) or 1.0
        for name, s in corruptions.conditions(grid):
            bad = corruptions.corrupt(image, spacing, name, s, grid, key=f"{p}|{phase}")
            rows.append({"corruption": name, "level": s, "change_pct": 100 * float(np.abs(bad - image).mean()) / rng_})
table = pd.DataFrame(rows).groupby(["corruption", "level"])["change_pct"].mean().unstack().round(2)
table

## How to choose

- **Level 1** should be barely visible. **Level 5** should be clearly degraded, with the heart still recognisable to a person.
- Each row of the table should **increase from left to right**. If it does not, the values are not a severity scale.
- The four corruptions do not need identical numbers, but one should not be ten times stronger than another at the same level.
- Keep within what a real scanner could plausibly produce. The point is a proxy for scanner differences, not a stress test.

When you are both satisfied:
1. Edit the values in `configs/severity_grid.yaml`.
2. Re-run this notebook to confirm.
3. Follow `docs/FREEZE_CHECKLIST.md`: set `status: frozen`, commit, tag.

After that the values do not change.